In [60]:
import pyautogui as pyg
pyg.sleep(2)
basex = 1284
basey = 407
gap = 40

def gem_position(r, c):
    return (basex + c * gap, basey + r * gap)


KeyboardInterrupt: 

In [21]:
print(gem_position(0, 0))
print(gem_position(0, 1))
print(gem_position(0, 2))

(1284, 407)
(1324, 407)
(1364, 407)


In [ ]:
im = pyg.screenshot()
for i in range(8):
    x, y = gem_position(0, i)
    pyg.click(x, y, duration=1)

In [ ]:
def get_gem_color(r, c, jm):
    """從 PyAutoGUI 的全螢幕截圖取得指定寶石中心的 RGB 顏色。"""
    x, y = gem_position(r, c)
    return jm.getpixel((x, y))

In [ ]:
def can_move_left(r, c, im):
    """判斷向左交換後，兩顆寶石是否形成水平或垂直三連線。"""
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    current = get_gem_color(r, c, im)
    left = get_gem_color(r, c - 1, im)
    if current == left:
        return False

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return left
        if (row, col) == (r, c - 1):
            return current
        return get_gem_color(row, col, im)

    for row, col in ((r, c), (r, c - 1)):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr, nc = row + dr * direction, col + dc * direction
                while 0 <= nr < 8 and 0 <= nc < 8 and color_after_swap(nr, nc) == color:
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
    return False

In [ ]:
def can_move_right(r, c, im):
    if not (0 <= r < 8 and 0 <= c < 7):
        return False
    return can_move_left(r, c + 1, im)


def can_move_up(r, c, im):
    if not (1 <= r < 8 and 0 <= c < 8):
        return False

    current = get_gem_color(r, c, im)
    above = get_gem_color(r - 1, c, im)
    if current == above:
        return False

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return above
        if (row, col) == (r - 1, c):
            return current
        return get_gem_color(row, col, im)

    for row, col in ((r, c), (r - 1, c)):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr, nc = row + dr * direction, col + dc * direction
                while 0 <= nr < 8 and 0 <= nc < 8 and color_after_swap(nr, nc) == color:
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
    return False


def can_move_down(r, c, im):
    if not (0 <= r < 7 and 0 <= c < 8):
        return False
    return can_move_up(r + 1, c, im)

In [ ]:
def find_moves(im):
    """列出截圖中可能形成三連線的交換，每組交換只列一次。"""
    moves = []
    for r in range(8):
        for c in range(8):
            if can_move_right(r, c, im):
                moves.append(((r, c), (r, c + 1)))
            if can_move_down(r, c, im):
                moves.append(((r, c), (r + 1, c)))
    return moves

# 執行此儲存格時截取一次畫面，僅列出結果，不操作滑鼠。
im = pyg.screenshot()
moves = find_moves(im)
print('找到', len(moves), '個可能的交換：')
for start, end in moves:
    print(start, '→', end)

In [ ]:
import time

# 執行前校正棋盤座標；將滑鼠移至螢幕左上角可觸發 PyAutoGUI 緊急停止。
pyg.FAILSAFE = True
start_time = time.monotonic()
im = pyg.screenshot()
directions = ((can_move_left, 0, -1), (can_move_right, 0, 1),
              (can_move_up, -1, 0), (can_move_down, 1, 0))

try:
    while time.monotonic() - start_time < 60:
        moved = False
        for r in range(8):
            if moved or time.monotonic() - start_time >= 60:
                break
            for c in range(8):
                if moved or time.monotonic() - start_time >= 60:
                    break
                for can_move, dr, dc in directions:
                    if time.monotonic() - start_time >= 60:
                        break
                    if can_move(r, c, im):
                        pyg.moveTo(*gem_position(r, c))
                        pyg.dragTo(*gem_position(r + dr, c + dc), duration=0.2, button='left')
                        moved = True
                        break
        # 等待畫面變化，再以新截圖進行下一輪判斷。
        time.sleep(0.5 if moved else 0.2)
        im = pyg.screenshot()
except pyg.FailSafeException:
    print('已觸發滑鼠緊急停止。')
finally:
    print('自動操作已停止。')